In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

# Find the CSV inside the kagglehub downloaded folder
csv_files = []
for root, _, files in os.walk(path):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, f))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV found inside the downloaded dataset folder!")

csv_path = csv_files[0]  # take the first CSV
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)
df


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
import numpy as np

# Detect target column
possible_targets = ["target", "label", "class", "y"]
target_col = None
for t in possible_targets:
    if t in df.columns:
        target_col = t
        break
if target_col is None:
    target_col = df.columns[-1]  # fallback

print("Target column =", target_col)

# Separate features for cleaning
feature_cols = [c for c in df.columns if c != target_col]

num_cols = df[feature_cols].select_dtypes(include=["int64","float64"]).columns.tolist()
cat_cols = df[feature_cols].select_dtypes(include=["object","category","bool"]).columns.tolist()

# Fill missing in numeric with median, categorical with mode
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0] if not df[c].mode().empty else "Unknown")

# If target has missing values, drop those rows
df = df.dropna(subset=[target_col])

df.isna().sum()


In [ ]:
# Task 2: Write your code here:
# One-hot encode categorical features if any
dup = df.duplicated().sum()
print("Duplicates:", dup)

if dup > 0:
    df = df.drop_duplicates()

print("Duplicates after drop:", df.duplicated().sum())


In [ ]:
# Task 3: Write your code here:
# One-hot encode categorical features if any
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df_encoded.head()


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("X_scaled shape:", X_scaled.shape, "| y shape:", y.shape)


In [ ]:
# Task 5: Write your code here:
y_series = pd.Series(y)
counts = y_series.value_counts()
print("Class counts:\n", counts)

majority_ratio = counts.max() / counts.sum()
is_binary = counts.shape[0] == 2

print("\nMajority class ratio:", round(float(majority_ratio), 3))
if is_binary and majority_ratio > 0.60:
    print("✅ Target looks IMBALANCED (binary).")
else:
    print("✅ Target does NOT look severely imbalanced (or not binary).")


In [ ]:
# Task 1: Write your code here:
# X_scaled and y already prepared above
# (Keeping this cell for clarity)
X_data = X_scaled
y_data = y


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from CatBoost import CatBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []
metric_name = "F1" if (is_binary and majority_ratio > 0.60) else "Accuracy"
print("Using metric:", metric_name)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_data), start=1):
    X_train, X_val = X_data[train_idx], X_data[val_idx]
    y_train, y_val = y_data[train_idx], y_data[val_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=0
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    if metric_name == "F1":
        score = f1_score(y_val, preds, average="binary")
    else:
        score = accuracy_score(y_val, preds)

    scores.append(score)
    print(f"Fold {fold} {metric_name}: {score:.4f}")

print(f"\nAverage {metric_name} across folds: {np.mean(scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

final_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=0
)
final_model.fit(X_data, y_data)

importances = final_model.get_feature_importance()
feature_names = X.columns

# Plot top 15
top_k = 15
top_idx = np.argsort(importances)[-top_k:]

plt.figure(figsize=(10,6))
plt.barh(feature_names[top_idx], importances[top_idx])
plt.title("Top Feature Importances (CatBoost)")
plt.xlabel("Importance")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_idx = int(np.argmax(importances))
golden_feature = feature_names[golden_idx]
print("Golden Feature:", golden_feature)
print("Importance:", importances[golden_idx])

In [ ]:
# Task Bonus: Write your code here:
# Create X with only golden feature (single column)
X_golden = df_encoded[[golden_feature]].values

# Scale it (important even for 1 feature)
scaler_g = StandardScaler()
X_golden_scaled = scaler_g.fit_transform(X_golden)

scores_golden = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_golden_scaled, y_data), start=1):
    X_train, X_val = X_golden_scaled[train_idx], X_golden_scaled[val_idx]
    y_train, y_val = y_data[train_idx], y_data[val_idx]

    model_g = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=0
    )

    model_g.fit(X_train, y_train)
    preds = model_g.predict(X_val)

    if metric_name == "F1":
        score = f1_score(y_val, preds, average="binary")
    else:
        score = accuracy_score(y_val, preds)

    scores_golden.append(score)
    print(f"Fold {fold} Golden-only {metric_name}: {score:.4f}")

print(f"\nAverage Golden-only {metric_name}: {np.mean(scores_golden):.4f}")
print(f"Full-model Average {metric_name}: {np.mean(scores):.4f}")
